# Task 4 — Sentiment Analysis

## Objective
Analyse text reviews and classify them as **positive** or **negative** using Natural Language Processing (NLP) and machine learning.

## Dataset
This project uses the public **NLTK Movie Reviews** corpus. It contains 2,000 labelled movie reviews:
- 1,000 positive
- 1,000 negative

The notebook downloads the corpus automatically with NLTK, so a large dataset file does not need to be manually uploaded.

## Workflow
1. Load the labelled review corpus.
2. Inspect class distribution and text length.
3. Clean and normalize review text.
4. Convert text into TF-IDF features.
5. Split into training and testing data.
6. Train Logistic Regression.
7. Evaluate accuracy, precision, recall and F1-score.
8. Visualize the confusion matrix.
9. Identify important positive and negative words.
10. Test the model on new example reviews.

**Tools:** Python, Pandas, NumPy, NLTK, Scikit-learn, Matplotlib, Seaborn.


In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Download the public corpus if it is not already installed
nltk.download("movie_reviews", quiet=True)

print("Movie review files:", len(movie_reviews.fileids()))


## 1. Load and inspect the dataset

The NLTK Movie Reviews corpus provides a predefined sentiment label through the folder/category:
- `pos` → Positive
- `neg` → Negative


In [ ]:
documents = []
labels = []

for fileid in movie_reviews.fileids():
    words = movie_reviews.words(fileid)
    text = " ".join(words)
    label = movie_reviews.categories(fileid)[0]
    documents.append(text)
    labels.append(label)

df = pd.DataFrame({"text": documents, "sentiment": labels})

print("Shape:", df.shape)
display(df.head())
print("\nClass distribution:")
display(df["sentiment"].value_counts())
print("\nMissing values:")
display(df.isna().sum())


In [ ]:
df["word_count"] = df["text"].str.split().str.len()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="sentiment")
plt.title("Positive vs Negative Reviews")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="word_count", hue="sentiment", bins=30, element="step")
plt.title("Review Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Number of Reviews")
plt.show()


**Observation:** The corpus is balanced, with the same number of positive and negative reviews. Review lengths vary considerably, so the model must focus on informative words and phrases rather than raw document length.


## 2. Text preprocessing

The cleaning function:
- converts text to lowercase
- removes URLs
- removes punctuation and numbers
- removes extra whitespace

Stop-word handling is performed by the TF-IDF vectorizer.


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

display(df[["text", "clean_text", "sentiment"]].head())


## 3. Train-test split and TF-IDF

TF-IDF converts text into numerical features. It gives higher importance to words that are useful in a document but less common across the whole collection.

A stratified 80/20 split keeps the positive/negative class balance in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["sentiment"],
    test_size=0.20,
    random_state=42,
    stratify=df["sentiment"]
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training samples:", X_train_tfidf.shape[0])
print("Testing samples:", X_test_tfidf.shape[0])
print("TF-IDF features:", X_train_tfidf.shape[1])


## 4. Train Logistic Regression

Logistic Regression is a strong and interpretable baseline for binary text classification, especially when combined with TF-IDF features.


In [ ]:
model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train_tfidf, y_train)

pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, pos_label="pos")
recall = recall_score(y_test, pred, pos_label="pos")
f1 = f1_score(y_test, pred, pos_label="pos")

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics)
print(classification_report(y_test, pred, target_names=["Negative", "Positive"]))


## 5. Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, pred, labels=["neg", "pos"])

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


**Observation:** The confusion matrix shows how many reviews were correctly classified and where the model made mistakes. Accuracy alone is not enough, so precision, recall and F1-score are also reported.


## 6. Most influential words

Logistic Regression coefficients can be inspected to understand which words push predictions toward positive or negative sentiment.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]

positive_idx = np.argsort(coefficients)[-15:][::-1]
negative_idx = np.argsort(coefficients)[:15]

top_positive = pd.DataFrame({
    "word": feature_names[positive_idx],
    "coefficient": coefficients[positive_idx]
})

top_negative = pd.DataFrame({
    "word": feature_names[negative_idx],
    "coefficient": coefficients[negative_idx]
})

print("Top positive words:")
display(top_positive)

print("Top negative words:")
display(top_negative)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_positive.sort_values("coefficient").plot(
    kind="barh", x="word", y="coefficient", ax=axes[0], legend=False
)
axes[0].set_title("Words Associated with Positive Sentiment")
axes[0].set_xlabel("Model coefficient")

top_negative.sort_values("coefficient").plot(
    kind="barh", x="word", y="coefficient", ax=axes[1], legend=False
)
axes[1].set_title("Words Associated with Negative Sentiment")
axes[1].set_xlabel("Model coefficient")

plt.tight_layout()
plt.show()


## 7. Test new reviews

The same cleaning and TF-IDF pipeline can be applied to unseen text.


In [ ]:
new_reviews = [
    "This movie was excellent, emotional and beautifully acted.",
    "The story was boring, predictable and disappointing.",
    "A brilliant performance with a great story and memorable scenes.",
    "I did not enjoy this movie and the ending was terrible."
]

new_clean = [clean_text(x) for x in new_reviews]
new_features = vectorizer.transform(new_clean)
new_predictions = model.predict(new_features)
new_probabilities = model.predict_proba(new_features).max(axis=1)

predictions_df = pd.DataFrame({
    "Review": new_reviews,
    "Predicted Sentiment": ["Positive" if x == "pos" else "Negative" for x in new_predictions],
    "Confidence": np.round(new_probabilities, 3)
})

display(predictions_df)


## Conclusion

This project demonstrates a complete NLP sentiment-analysis workflow:
- labelled text data was loaded and inspected
- reviews were cleaned
- TF-IDF transformed text into machine-learning features
- Logistic Regression classified sentiment
- accuracy, precision, recall and F1-score were calculated
- a confusion matrix was visualized
- important sentiment-associated words were extracted
- the model was tested on new reviews

### Business use
The same approach can be adapted to analyse customer reviews, social-media comments, product feedback and survey responses to understand overall customer sentiment at scale.

### Limitation
Movie-review sentiment does not represent every domain. Sarcasm, context, slang and domain-specific language can cause classification errors. The model should therefore be evaluated on the target business's own text before deployment.


In [ ]:
# Save the evaluation results
metrics.to_csv("../output/model_metrics.csv", index=False)
predictions_df.to_csv("../output/example_predictions.csv", index=False)

print("Task 4 sentiment analysis completed.")
